In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

SetUp Cell


In [1]:
import os
import gdown
import zipfile
import shutil

os.makedirs("/kaggle/working/data", exist_ok=True)
os.makedirs("/kaggle/working/models", exist_ok=True)
os.makedirs("/kaggle/working/results", exist_ok=True)

# Download CSV
print("Downloading CSV...")
gdown.download(
    "https://drive.google.com/uc?id=1m5D9X5xBNfd25kMMB9G0CBPDWcsMc0Ye",
    "/kaggle/working/data/clean_multimodal_samples.csv",
    quiet=False
)
print(f"✓ CSV downloaded")

# Check space before downloading ZIP
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\nDisk space free: {free//(1024**3)} GB")

Downloading...
From: https://drive.google.com/uc?id=1m5D9X5xBNfd25kMMB9G0CBPDWcsMc0Ye
To: /kaggle/working/data/clean_multimodal_samples.csv
100%|██████████| 28.2M/28.2M [00:00<00:00, 69.7MB/s]

✓ CSV downloaded

Disk space free: 19 GB


Step 3 — Download + Extract Images****

In [2]:
# Download ZIP
print("Downloading ZebraMap images ZIP...")
gdown.download(
    "https://drive.google.com/uc?id=1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq",
    "/kaggle/working/zebramap_final.zip",
    quiet=False
)
print("✓ ZIP downloaded")

# Extract to temp
print("\nExtracting to /kaggle/temp/images...")
os.makedirs("/kaggle/temp/images", exist_ok=True)

with zipfile.ZipFile("/kaggle/working/zebramap_final.zip", 'r') as z:
    total = len(z.namelist())
    for i, member in enumerate(z.namelist()):
        z.extract(member, "/kaggle/temp/images")
        if i % 10000 == 0:
            print(f"  {i}/{total} ({i/total*100:.0f}%)")

print("✓ Extraction complete")

# Free space — delete ZIP
os.remove("/kaggle/working/zebramap_final.zip")
print("✓ ZIP deleted — space freed")

# Verify
total_imgs = sum(
    len(files) for _, _, files in os.walk("/kaggle/temp/images"))
print(f"✓ Total images: {total_imgs}")

Downloading...
From (original): https://drive.google.com/uc?id=1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq
From (redirected): https://drive.google.com/uc?id=1KVsV08Mh8vXCQNu0V1Tb0qUzNorweEnq&confirm=t&uuid=3f4f846e-b182-4e30-af71-139675d3e03f
To: /kaggle/working/zebramap_final.zip
100%|██████████| 10.5G/10.5G [01:50<00:00, 95.5MB/s]


✓ ZIP downloaded

Extracting to /kaggle/temp/images...
  0/110262 (0%)
  10000/110262 (9%)
  20000/110262 (18%)
  30000/110262 (27%)
  40000/110262 (36%)
  50000/110262 (45%)
  60000/110262 (54%)
  70000/110262 (63%)
  80000/110262 (73%)
  90000/110262 (82%)
  100000/110262 (91%)
  110000/110262 (100%)
✓ Extraction complete
✓ ZIP deleted — space freed
✓ Total images: 79478


Prepare Data + Train CNN**

In [3]:
import pandas as pd
import ast
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from PIL import Image
import json
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {torch.cuda.get_device_name(0)}")

# Load + prepare data
df = pd.read_csv("/kaggle/working/data/clean_multimodal_samples.csv")

# Fix image paths
def fix_path(img_val):
    try:
        images = eval(img_val)
        fixed = []
        for img in images:
            old = img['path']
            if 'images/' in old:
                rel = old.split('images/')[-1]
                img['path'] = f"/kaggle/temp/images/{rel}"
            fixed.append(img)
        return fixed
    except:
        return []

print("Fixing image paths...")
df['images_fixed'] = df['images'].apply(fix_path)

# Encode labels
new_le = LabelEncoder()
df['label'] = new_le.fit_transform(df['disease_name'])

# Filter + split
label_counts = df['label'].value_counts()
valid_labels  = label_counts[label_counts >= 2].index
df_filtered   = df[df['label'].isin(valid_labels)].reset_index(drop=True)

train_df, test_df = train_test_split(
    df_filtered, test_size=0.2,
    random_state=42, stratify=df_filtered['label']
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

all_labels    = sorted(train_df['label'].unique())
label_remap   = {old: new for new, old in enumerate(all_labels)}
reverse_remap = {new: old for old, new in label_remap.items()}
NUM_CLASSES   = len(all_labels)

train_df['label'] = train_df['label'].map(label_remap)
test_df['label']  = test_df['label'].map(label_remap)
test_df = test_df.dropna(subset=['label']).reset_index(drop=True)
test_df['label']  = test_df['label'].astype(int)

# Verify images
found = 0
for _, row in train_df.head(20).iterrows():
    imgs = row['images_fixed']
    if imgs and os.path.exists(imgs[0]['path']):
        found += 1

print(f"✓ Data ready")
print(f"  Train   : {len(train_df)}")
print(f"  Test    : {len(test_df)}")
print(f"  Classes : {NUM_CLASSES}")
print(f"  Images  : {found}/20 verified")

✓ Device: Tesla T4
Fixing image paths...
✓ Data ready
  Train   : 29049
  Test    : 7263
  Classes : 1199
  Images  : 16/20 verified


CNN Dataset + Training

In [4]:
class ZebraMapImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.samples   = []
        self.transform = transform

        for _, row in df.iterrows():
            try:
                images = row['images_fixed']
                if not isinstance(images, list):
                    images = eval(images)
                for img_info in images:
                    path = img_info['path']
                    if os.path.exists(path):
                        self.samples.append({
                            'path' : path,
                            'label': row['label']
                        })
                        break
            except:
                continue

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        try:
            img = Image.open(sample['path']).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        return {
            'image': img,
            'label': torch.tensor(sample['label'], dtype=torch.long)
        }

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("Building datasets...")
cnn_train_ds = ZebraMapImageDataset(train_df, train_transform)
cnn_test_ds  = ZebraMapImageDataset(test_df,  test_transform)

cnn_train_loader = DataLoader(cnn_train_ds, batch_size=64,
                               shuffle=True,  num_workers=4)
cnn_test_loader  = DataLoader(cnn_test_ds,  batch_size=64,
                               shuffle=False, num_workers=4)

print(f"✓ Datasets ready")
print(f"  Train images  : {len(cnn_train_ds)}")
print(f"  Test images   : {len(cnn_test_ds)}")
print(f"  Train batches : {len(cnn_train_loader)}")

class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes, dropout=0.4):
        super().__init__()
        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1)
        for layer in list(backbone.children())[:-3]:
            for param in layer.parameters():
                param.requires_grad = False
        self.features   = nn.Sequential(
            *list(backbone.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

CNN_EPOCHS = 15
cnn_model  = ResNet50Classifier(NUM_CLASSES).to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = AdamW(
    filter(lambda p: p.requires_grad, cnn_model.parameters()),
    lr=1e-4, weight_decay=0.01
)
scheduler  = CosineAnnealingLR(
    optimizer, T_max=CNN_EPOCHS, eta_min=1e-6)

cnn_losses = []
cnn_accs   = []

print(f"\nTraining CNN — {CNN_EPOCHS} epochs")
print(f"  Images  : {len(cnn_train_ds)}")
print(f"  Classes : {NUM_CLASSES}")
print("-" * 55)

for epoch in range(CNN_EPOCHS):
    cnn_model.train()
    total_loss, correct, total = 0, 0, 0

    for batch in cnn_train_loader:
        images = batch['image'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = cnn_model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            cnn_model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    scheduler.step()
    avg_loss = total_loss / len(cnn_train_loader)
    acc      = correct / total * 100
    cnn_losses.append(avg_loss)
    cnn_accs.append(acc)
    print(f"Epoch {epoch+1:02d}/{CNN_EPOCHS} | "
          f"Loss: {avg_loss:.4f} | "
          f"Train Acc: {acc:.2f}%")

print("-" * 55)
print("✓ CNN training complete")

Building datasets...
✓ Datasets ready
  Train images  : 24527
  Test images   : 6135
  Train batches : 384
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 210MB/s]



Training CNN — 15 epochs
  Images  : 24527
  Classes : 1199
-------------------------------------------------------
Epoch 01/15 | Loss: 6.8809 | Train Acc: 2.28%
Epoch 02/15 | Loss: 6.1730 | Train Acc: 5.38%
Epoch 03/15 | Loss: 5.7787 | Train Acc: 6.74%
Epoch 04/15 | Loss: 5.4714 | Train Acc: 8.42%
Epoch 05/15 | Loss: 5.1786 | Train Acc: 10.80%
Epoch 06/15 | Loss: 4.9239 | Train Acc: 12.38%
Epoch 07/15 | Loss: 4.6652 | Train Acc: 15.38%
Epoch 08/15 | Loss: 4.4179 | Train Acc: 17.67%
Epoch 09/15 | Loss: 4.1965 | Train Acc: 20.77%
Epoch 10/15 | Loss: 3.9990 | Train Acc: 23.17%
Epoch 11/15 | Loss: 3.8264 | Train Acc: 25.62%
Epoch 12/15 | Loss: 3.6996 | Train Acc: 27.72%
Epoch 13/15 | Loss: 3.6082 | Train Acc: 29.06%
Epoch 14/15 | Loss: 3.5445 | Train Acc: 30.07%
Epoch 15/15 | Loss: 3.5113 | Train Acc: 30.92%
-------------------------------------------------------
✓ CNN training complete


Evaluate + Save Results

In [5]:
def evaluate_topk(model, loader, device, k=5):
    model.eval()
    all_preds, all_labels = [], []
    top5_correct, total   = 0, 0

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            logits = model(images)

            preds  = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            k_actual = min(k, logits.size(1))
            topk     = logits.topk(k_actual, dim=1).indices
            for i, lbl in enumerate(labels):
                if lbl in topk[i]:
                    top5_correct += 1
            total += labels.size(0)

    return {
        "accuracy"     : round(accuracy_score(
                            all_labels, all_preds)*100, 2),
        "f1_macro"     : round(f1_score(
                            all_labels, all_preds,
                            average='macro',
                            zero_division=0)*100, 2),
        "f1_weighted"  : round(f1_score(
                            all_labels, all_preds,
                            average='weighted',
                            zero_division=0)*100, 2),
        "top5_accuracy": round(top5_correct/total*100, 2),
        "total_samples": total
    }

print("Evaluating CNN...")
cnn_metrics = evaluate_topk(cnn_model, cnn_test_loader, device)

# NLP metrics from morning (already saved to Drive)
nlp_metrics = {
    "accuracy"     : 16.56,
    "f1_macro"     : 2.75,
    "f1_weighted"  : 10.84,
    "top5_accuracy": 35.98,
    "total_samples": 7263
}

print("\n" + "=" * 55)
print("EXP 4 — ZEBRAMAP BASELINE COMPLETE")
print("=" * 55)
print(f"  NLP — Accuracy : {nlp_metrics['accuracy']}%")
print(f"  NLP — Top-5    : {nlp_metrics['top5_accuracy']}%")
print(f"  CNN — Accuracy : {cnn_metrics['accuracy']}%")
print(f"  CNN — F1 Macro : {cnn_metrics['f1_macro']}%")
print(f"  CNN — Top-5    : {cnn_metrics['top5_accuracy']}%")

# Save CNN model
torch.save({
    'model_state_dict': cnn_model.state_dict(),
    'label_remap'     : label_remap,
    'reverse_remap'   : reverse_remap,
    'num_classes'     : NUM_CLASSES,
    'metrics'         : cnn_metrics
}, "/kaggle/working/models/exp4_cnn_baseline.pt")
print(f"\n✓ CNN model saved")

# Save summary JSON
summary = {
    "day"        : 7,
    "experiment" : "exp4_zebramap_baseline",
    "nlp_metrics": nlp_metrics,
    "cnn_metrics": cnn_metrics,
    "status"     : "Day 7 complete"
}

with open("/kaggle/working/results/day7_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("✓ Summary saved")
print(f"\nDAY 7 COMPLETE ✓")
print(f"  Next → Day 8: Results dashboard")

Evaluating CNN...

EXP 4 — ZEBRAMAP BASELINE COMPLETE
  NLP — Accuracy : 16.56%
  NLP — Top-5    : 35.98%
  CNN — Accuracy : 10.2%
  CNN — F1 Macro : 2.77%
  CNN — Top-5    : 25.7%

✓ CNN model saved
✓ Summary saved

DAY 7 COMPLETE ✓
  Next → Day 8: Results dashboard
